In [1]:
import re
import math
import random
from collections import Counter, defaultdict

In [2]:
def to_sentences(text):
    return [
        s.strip()
        for s in re.split(r"[.!?]+", text)
        if s.strip()
    ]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

In [3]:
with open("corpus.txt", "r") as file:
    sents = to_sentences(file.read())

print("Number of sentences:", len(sents))

Number of sentences: 60


In [4]:
def build(sents, n):
    """Map each (n-1)-word context to a Counter of following words."""
    foll = defaultdict(Counter)

    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]

        for i in range(len(toks) - n + 1):
            context = tuple(toks[i:i + n - 1])
            next_word = toks[i + n - 1]
            foll[context][next_word] += 1

    return foll

foll = build(sents, n=2)

In [5]:
V = set(
    token
    for sentence in sents
    for token in to_tokens(sentence)
) | {"</s>"}

print("Vocabulary size:", len(V))
print("Contains </s>:", "</s>" in V)

Vocabulary size: 170
Contains </s>: True


In [6]:
def prob_mle(ctx, w):
    c = foll[(ctx,)]
    total = sum(c.values())

    return c[w] / total if total else 0.0

In [7]:
def prob_addk(ctx, w, k=1.0):
    c = foll[(ctx,)]
    total = sum(c.values())

    return (c[w] + k) / (total + k * len(V))

In [8]:
print(
    "MLE   P(coffee | drinks) =",
    prob_mle("drinks", "coffee")
)

print(
    "add-1 P(coffee | drinks) =",
    round(prob_addk("drinks", "coffee"), 5)
)

print(
    "add-1 P(green  | drinks) =",
    round(prob_addk("drinks", "green"), 5)
)

MLE   P(coffee | drinks) = 0.0
add-1 P(coffee | drinks) = 0.00581
add-1 P(green  | drinks) = 0.01163


In [9]:
for k in [0.01, 0.1, 0.5, 1.0, 2.0]:
    p_coffee = prob_addk("drinks", "coffee", k)
    p_green = prob_addk("drinks", "green", k)

    print(
        f"k={k:<4} | "
        f"P(coffee | drinks)={p_coffee:.5f} | "
        f"P(green | drinks)={p_green:.5f}"
    )

k=0.01 | P(coffee | drinks)=0.00270 | P(green | drinks)=0.27297
k=0.1  | P(coffee | drinks)=0.00526 | P(green | drinks)=0.05789
k=0.5  | P(coffee | drinks)=0.00575 | P(green | drinks)=0.01724
k=1.0  | P(coffee | drinks)=0.00581 | P(green | drinks)=0.01163
k=2.0  | P(coffee | drinks)=0.00585 | P(green | drinks)=0.00877


In [10]:
context = "drinks"
k = 1.0

total_probability = sum(
    prob_addk(context, word, k)
    for word in V
)

print("Total probability:", round(total_probability, 6))

Total probability: 1.0


In [11]:
print("MLE:", prob_mle("spaceship", "coffee"))
print("Add-1:", prob_addk("spaceship", "coffee"))

MLE: 0.0
Add-1: 0.0058823529411764705


## Observations — Lesson 5

1. MLE assigns zero probability to word pairs that never occur
   in the training corpus.

2. A single zero probability can make the probability of an
   entire sentence zero.

3. Add-k smoothing adds a small constant to every word count,
   giving unseen word combinations non-zero probabilities.

4. Laplace smoothing is the special case where k = 1.

5. Increasing k assigns more probability mass to unseen words,
   but can weaken the influence of observed frequencies.

6. Smoothing ensures that probabilities across the vocabulary
   sum to 1 for a given context.

7. Smoothing solves zero probabilities, but it does not
   automatically make an unseen word combination linguistically
   sensible.